In [ ]:
!pip -q install langchain
!pip -q install langchain-community
!pip -q install langchain-google-genai
!pip -q install faiss-cpu
!pip -q install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 77.1 MB/s eta 0:00:00


#Libraries

In [ ]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough

from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_1623/286147452.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

#Load Data

In [ ]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 18.2 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
document=PyPDFLoader("/content/RNN.pdf")
docs=document.load()
print(len(docs))

42


In [ ]:
docs[0].page_content

'– Understanding LSTM –\na tutorial into Long Short-Term Memory\nRecurrent Neural Networks\nRalf C. Staudemeyer\nFaculty of Computer Science\nSchmalkalden University of Applied Sciences, Germany\nE-Mail: r.staudemeyer@hs-sm.de\nEric Rothstein Morris\n(Singapore University of Technology and Design, Singapore\nE-Mail: eric rothstein@sutd.edu.sg)\nSeptember 23, 2019\nAbstract\nLong Short-Term Memory Recurrent Neural Networks (LSTM-RNN)\nare one of the most powerful dynamic classiﬁers publicly known. The net-\nwork itself and the related learning algorithms are reasonably well docu-\nmented to get an idea how it works. This paper will shed more light into\nunderstanding how LSTM-RNNs evolved and why they work impressively\nwell, focusing on the early, ground-breaking publications. We signiﬁcantly\nimproved documentation and ﬁxed a number of errors and inconsistencies\nthat accumulated in previous publications. To support understanding we\nas well revised and uniﬁed the notation used.\n1 In

# Chunking

In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks=splitter.split_documents(docs)

In [ ]:
chunks[0]

Document(metadata={'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-09-23T00:41:49+00:00', 'author': 'Ralf C. Staudemeyer, Eric Rothstein Morris', 'keywords': '', 'moddate': '2019-09-23T00:41:49+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': 'Understanding Long Short-Term Memory Recurrent Neural Networks – a tutorial-like introduction', 'trapped': '/False', 'source': '/content/RNN.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1'}, page_content='– Understanding LSTM –\na tutorial into Long Short-Term Memory\nRecurrent Neural Networks\nRalf C. Staudemeyer\nFaculty of Computer Science\nSchmalkalden University of Applied Sciences, Germany\nE-Mail: r.staudemeyer@hs-sm.de\nEric Rothstein Morris\n(Singapore University of Technology and Design, Singapore\nE-Mail: eric rothstein@sutd.edu.sg)\nSeptember 23, 2019\nAbstract\nLong Short-Term Memory Recurrent

In [ ]:
chunks[1].page_content

'that accumulated in previous publications. To support understanding we\nas well revised and uniﬁed the notation used.\n1 Introduction\nThis article is an tutorial-like introduction initially developed as supplementary\nmaterial for lectures focused on Artiﬁcial Intelligence. The interested reader\ncan deepen his/her knowledge by understanding Long Short-Term Memory Re-\ncurrent Neural Networks (LSTM-RNN) considering its evolution since the early\nnineties. Todays publications on LSTM-RNN use a slightly diﬀerent notation\nand a much more summarized representation of the derivations. Nevertheless\nthe authors found the presented approach very helpful and we are conﬁdent this\npublication will ﬁnd its audience.\nMachine learning is concerned with the development of algorithms that au-\ntomatically improve by practice. Ideally, the more the learning algorithm is run,\nthe better the algorithm becomes. It is the task of the learning algorithm to'

In [ ]:
chunks[0].metadata

{'producer': 'pdfTeX-1.40.17',
 'creator': 'LaTeX with hyperref package',
 'creationdate': '2019-09-23T00:41:49+00:00',
 'author': 'Ralf C. Staudemeyer, Eric Rothstein Morris',
 'keywords': '',
 'moddate': '2019-09-23T00:41:49+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2',
 'subject': '',
 'title': 'Understanding Long Short-Term Memory Recurrent Neural Networks – a tutorial-like introduction',
 'trapped': '/False',
 'source': '/content/RNN.pdf',
 'total_pages': 42,
 'page': 0,
 'page_label': '1'}

# Embedding

In [ ]:
embedding=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_1623/3241142964.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# VectorDatabase

In [ ]:
db=FAISS.from_documents(
    documents=chunks,
    embedding=embedding
)

# Retriver

In [ ]:
retriver=db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":3,
        "fetch_k":12
    }
)

#LLM

In [ ]:
llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

#Prompt

In [ ]:
prompt=ChatPromptTemplate.from_template(
    """
You are an AI assistant.

Answer ONLY from the given context.

If the answer is not present,

say:

"I don't know."

Context:

{context}

Question:

{question}
"""
)

In [ ]:
chain=(
    {
        "context":retriver,
        "question":RunnablePassthrough()
    }
    |prompt
    |llm
    |StrOutputParser()
)

User Question

↓

Retriever

↓

Relevant Chunks

↓

Prompt

↓

LLM

↓

Output Parser

↓

Final Answer

In [ ]:
response=chain.invoke(
    "Explain Perceptron and Delta Learning Rule"
)
print(response)

Perceptron training is a type of supervised learning where the perceptron produces an output during training and compares it with a derived output value from the training data. If there is a misclassification, it modifies its weights accordingly. The perceptron will converge to reproduce the correct behavior in a finite time, provided that the training examples are linearly separable; convergence is not assured if they are not. Perceptrons can learn from samples, which are pairs of input and desired output ⟨x,d⟩. The weights of a trained perceptron should represent the line that separates linearly separable inputs.

The Delta Learning Rule is one of the most common training algorithms for perceptrons. It starts with random weights and guarantees convergence to an acceptable hypothesis. This rule modifies weights by using the gradient optimization descent algorithm, which alters them in the direction that produces the steepest descent along the error surface towards the global minimum e

In [ ]:
response=chain.invoke(
    "Explain Feed Forward"
)
response

'Feed-forward neural networks are loop-free and fully connected. This means that each neuron provides an input to each neuron in the following layer, and that none of the weights give an input to a neuron in a previous layer.'

In [ ]:
response=chain.invoke(
    "Tell About Turjo"
)
response

"I don't know."

#Runable step by step

In [31]:
docs=retriver.invoke("Explain Feed Forward")

In [32]:
for i,doc in enumerate(docs):
  print("="*30)
  print(f"Chunk: {i+1}")
  print(doc.page_content)

Chunk: 1
which are connected to other neurons, are called hidden neurons.
Feed-forward neural networks are loop-free and fully connected. This means
that each neuron provides an input to each neuron in the following layer, and
that none of the weights give an input to a neuron in a previous layer.
The simplest type of neural feed-forward networks are single-layer perceptron
networks. Single-layer neural networks consist of a set of input neurons, deﬁned
as the input layer, and a set of output neurons, deﬁned as the output layer. The
outputs of the input-layer neurons are directly connected to the neurons of the
output layer. The weights are applied to the connections between the input and
output layer.
In the single-layer perceptron network, every single perceptron calculates
the sum of the products of the weights and the inputs. The perceptron ﬁres ‘1’
if the value is above the threshold value; otherwise, the perceptron takes the
Chunk: 2
– Understanding LSTM –
a tutorial into Long Sh

In [35]:
[doc.metadata['page'] for doc in docs]

[7, 0, 40]

In [36]:
formatted_prompt=prompt.invoke(
    {
        "context":docs,
        "question":"Explain Feed Forward"
    }
)


In [38]:
formatted_prompt

ChatPromptValue(messages=[HumanMessage(content='\nYou are an AI assistant.\n\nAnswer ONLY from the given context.\n\nIf the answer is not present,\n\nsay:\n\n"I don\'t know."\n\nContext:\n\n[Document(id=\'e33dfd2b-8423-42db-a0ef-c88f55eceb7d\', metadata={\'producer\': \'pdfTeX-1.40.17\', \'creator\': \'LaTeX with hyperref package\', \'creationdate\': \'2019-09-23T00:41:49+00:00\', \'author\': \'Ralf C. Staudemeyer, Eric Rothstein Morris\', \'keywords\': \'\', \'moddate\': \'2019-09-23T00:41:49+00:00\', \'ptex.fullbanner\': \'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2\', \'subject\': \'\', \'title\': \'Understanding Long Short-Term Memory Recurrent Neural Networks – a tutorial-like introduction\', \'trapped\': \'/False\', \'source\': \'/content/RNN.pdf\', \'total_pages\': 42, \'page\': 7, \'page_label\': \'8\'}, page_content=\'which are connected to other neurons, are called hidden neurons.\\nFeed-forward neural networks are loop-free and fully

In [39]:
formatted_prompt.messages[0].content

'\nYou are an AI assistant.\n\nAnswer ONLY from the given context.\n\nIf the answer is not present,\n\nsay:\n\n"I don\'t know."\n\nContext:\n\n[Document(id=\'e33dfd2b-8423-42db-a0ef-c88f55eceb7d\', metadata={\'producer\': \'pdfTeX-1.40.17\', \'creator\': \'LaTeX with hyperref package\', \'creationdate\': \'2019-09-23T00:41:49+00:00\', \'author\': \'Ralf C. Staudemeyer, Eric Rothstein Morris\', \'keywords\': \'\', \'moddate\': \'2019-09-23T00:41:49+00:00\', \'ptex.fullbanner\': \'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2\', \'subject\': \'\', \'title\': \'Understanding Long Short-Term Memory Recurrent Neural Networks – a tutorial-like introduction\', \'trapped\': \'/False\', \'source\': \'/content/RNN.pdf\', \'total_pages\': 42, \'page\': 7, \'page_label\': \'8\'}, page_content=\'which are connected to other neurons, are called hidden neurons.\\nFeed-forward neural networks are loop-free and fully connected. This means\\nthat each neuron provi

In [40]:
response=llm.invoke(
    formatted_prompt
)
response

AIMessage(content='Feed-forward neural networks are loop-free and fully connected. This means that each neuron provides an input to each neuron in the following layer, and that none of the weights give an input to a neuron in a previous layer. The simplest type of neural feed-forward networks are single-layer perceptron networks.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f851f-fabe-7a91-b9fb-355631b03da7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1664, 'output_tokens': 392, 'total_tokens': 2056, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 329}})

In [41]:
response.content

'Feed-forward neural networks are loop-free and fully connected. This means that each neuron provides an input to each neuron in the following layer, and that none of the weights give an input to a neuron in a previous layer. The simplest type of neural feed-forward networks are single-layer perceptron networks.'

In [42]:
StrOutputParser().invoke(response)

'Feed-forward neural networks are loop-free and fully connected. This means that each neuron provides an input to each neuron in the following layer, and that none of the weights give an input to a neuron in a previous layer. The simplest type of neural feed-forward networks are single-layer perceptron networks.'

In [46]:
query="Explain Recurrent Neural Networks"
# step-1
docs=retriver.invoke(query)

#step-2
for doc in docs:
  print("="*50)
  print(doc.page_content)

# step-3
formatted_prompt=prompt.invoke(
    {
        "context":docs,
        "question":query
    }
)
#step-4
response=llm.invoke(formatted_prompt)
print("\nresponse:")
print(response)
print()

# step-5
final_ans=StrOutputParser().invoke(response)
print("\nFinal Response: ")
final_ans

– Understanding LSTM –
a tutorial into Long Short-Term Memory
Recurrent Neural Networks
Ralf C. Staudemeyer
Faculty of Computer Science
Schmalkalden University of Applied Sciences, Germany
E-Mail: r.staudemeyer@hs-sm.de
Eric Rothstein Morris
(Singapore University of Technology and Design, Singapore
E-Mail: eric rothstein@sutd.edu.sg)
September 23, 2019
Abstract
Long Short-Term Memory Recurrent Neural Networks (LSTM-RNN)
are one of the most powerful dynamic classiﬁers publicly known. The net-
work itself and the related learning algorithms are reasonably well docu-
mented to get an idea how it works. This paper will shed more light into
understanding how LSTM-RNNs evolved and why they work impressively
well, focusing on the early, ground-breaking publications. We signiﬁcantly
improved documentation and ﬁxed a number of errors and inconsistencies
that accumulated in previous publications. To support understanding we
as well revised and uniﬁed the notation used.
1 Introduction
[71] Oriol 

'Recurrent neural networks (RNNs) are dynamic systems that possess an internal state at each time step of the classification process. This characteristic stems from circular connections between higher- and lower-layer neurons, as well as optional self-feedback connections. These feedback connections enable RNNs to propagate data from earlier events to current processing steps, thereby building a memory of time series events.'